# L3.1 · ACT: de la grabación al ejemplo de entrenamiento

Antes de abrir la red, la pregunta más concreta: **¿qué le entra exactamente, y qué tiene que devolver?** Este notebook sigue un cuadro de una grabación hasta convertirse en el diccionario de tensores que entra al `forward` de ACT.


### El mapa de toda la serie

ACT es un **VAE condicional**:

- **Encoder del VAE** (sólo al entrenar): lee el estado y el bloque de acciones
  verdadero y lo resume en una variable `z`, el "estilo" de esa demostración.
- **Decoder = la policy** (al entrenar y al usar): un transformer
  encoder-decoder al estilo DETR. Lee la cámara, el estado y la `z`, y escribe
  el bloque entero de acciones de una pasada.
- **Al usarlo:** el encoder del VAE se tira y `z = 0`.

| Notebook | Qué abre |
|---|---|
| L3.1 | Los datos: de la grabación al ejemplo de entrenamiento |
| L3.2 | El encoder del VAE y la `z` |
| L3.3 | El decoder: la policy por dentro |
| L3.4 | El entrenamiento: un paso desarmado, y el modelo aprendiendo |
| L3.5 | El modelo en uso: ejecutar los bloques |

Todo usa el código real de LeRobot 0.4.1 y **nuestro modelo entrenado**
(`act_cheap_v1`), desarmado con `toolkit/policy/act_internals.py`. Sus tests
verifican que las piezas encadenadas dan exactamente lo mismo que el modelo.

### Los tamaños reales: paper, LeRobot y nuestro modelo

| | Paper (ALOHA) | LeRobot por defecto | **Nuestro modelo** |
|---|---|---|---|
| Cámaras | 4 de 480×640 | las que tenga el dataset | **1 de 360×640** |
| Tokens de imagen | 4 × 15×20 = 1200 | depende | **12×20 = 240** |
| Articulaciones | 14 | las del dataset | **26** |
| Tamaño de cada token (`dim_model`) | 512 | 512 | **192** |
| Cabezas de atención | 8 | 8 | **4** |
| Capa interna (`dim_feedforward`) | 3200 | 3200 | **768** |
| Capas del encoder | 4 | 4 | **4** |
| Capas del decoder | 7 declaradas, **1 usada** (bug) | 1 | **1** |
| Capas del encoder del VAE | 4 | 4 | **4** |
| `z` (`latent_dim`) | 32 | 32 | **32** |
| Bloque (`chunk_size`) | 100 | 100 | **48** |
| Peso del KL (β, `kl_weight`) | 10 | 10 | **10** |

Las capas del decoder: el código original de ACT tenía 7, pero por un error
sólo se usaba la primera. LeRobot pone 1 para replicar lo que realmente se
entrenaba ([issue](https://github.com/tonyzhaozh/act/issues/25)).

In [ ]:
import torch  # antes que lerobot: evita un import circular entre torchvision y torch._dynamo
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lerobot.datasets.lerobot_dataset import LeRobotDataset

from toolkit.paths import CHECKPOINTS_DIR, DATASETS_DIR, REPO_ROOT
from toolkit.policy import act_internals as act
from toolkit.policy.load import load_policy

plt.rcParams["figure.dpi"] = 110
torch.manual_seed(0)
ROOT = DATASETS_DIR / "local" / "pick_mameluco_act_v1"
CAMERA = "observation.images.cam_right_high"
loaded = load_policy(CHECKPOINTS_DIR / "act_cheap_v1" / "pretrained_model", device="cpu")
policy, model, config = loaded.policy, loaded.policy.model, loaded.policy.config
_ = policy.eval()

## 1. Un ejemplo = una observación + el bloque que vino después

Un cuadro del dataset tiene **una** acción. Pero ACT aprende a predecir las
**próximas 48**. El ejemplo de entrenamiento se arma así:

- **Entrada**: la observación del instante `t`, o sea la imagen y las 26
  articulaciones medidas.
- **Respuesta correcta**: las acciones de `t` a `t+47`, las órdenes que dio el
  operador durante los 1,6 s siguientes.

No hay un archivo con estos pares: se arman al vuelo. La configuración de la
política dice qué ventana necesita, y LeRobot la traduce a `delta_timestamps`.

In [ ]:
from lerobot.datasets.factory import resolve_delta_timestamps
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata

meta = LeRobotDatasetMetadata("local/pick_mameluco_act_v1", root=ROOT)
delta = resolve_delta_timestamps(config, meta)
print("ACT pide de 'action':", len(delta["action"]), "instantes, de", delta["action"][0], "a", round(delta["action"][-1], 3), "s")
print("de la observación pide sólo el presente (n_obs_steps =", config.n_obs_steps, ")")

dataset = LeRobotDataset("local/pick_mameluco_act_v1", root=ROOT, video_backend="pyav", delta_timestamps=delta)
sample = dataset[3000]
{k: (tuple(v.shape), str(v.dtype)) if hasattr(v, "shape") else v for k, v in sample.items()}

Lo vemos sobre la línea de tiempo del episodio: la imagen de `t`, y el bloque que la red tiene que predecir desde ahí.

In [ ]:
names = meta.features["action"]["names"][0]
episode = int(sample["episode_index"])
start = int(meta.episodes[episode]["dataset_from_index"])
end = int(meta.episodes[episode]["dataset_to_index"])
t = int(sample["index"]) - start
plain = LeRobotDataset("local/pick_mameluco_act_v1", root=ROOT, video_backend="pyav")
episode_actions = np.stack(plain.hf_dataset.with_format("numpy")[start:end]["action"])

fig = plt.figure(figsize=(15, 4))
ax_img = fig.add_subplot(1, 3, 1)
ax_img.imshow(sample[CAMERA].permute(1, 2, 0)); ax_img.set_title(f"observación en t = {t}"); ax_img.axis("off")
ax = fig.add_subplot(1, 3, (2, 3))
for joint in ["kRightShoulderPitch", "kRightElbow", "kRightHandIndex"]:
    j = names.index(joint)
    line, = ax.plot(episode_actions[:, j], alpha=0.35)
    ax.plot(range(t, t + 48), sample["action"][:, j], color=line.get_color(), lw=3, label=joint)
ax.axvline(t, color="k", ls="--"); ax.axvspan(t, t + 47, color="0.9")
ax.set_xlabel("cuadro del episodio"); ax.set_title("en grueso: el bloque que tiene que predecir (48 pasos)"); ax.legend(fontsize=8)
plt.tight_layout()

## 2. El borde del episodio: `action_is_pad`

Cerca del final del episodio, las 48 acciones siguientes no existen. LeRobot
repite la última y marca esas posiciones como **relleno**. El error de
entrenamiento las ignora.

¿Cuántos ejemplos tienen relleno?

In [ ]:
lengths = meta.episodes["length"]
padded_examples = sum(min(int(n), config.chunk_size - 1) for n in lengths)
print(f"{padded_examples} de {meta.total_frames} ejemplos ({padded_examples / meta.total_frames:.0%}) tienen algún paso de relleno")

near_end = dataset[end - 10]
print("a 10 cuadros del final:", int(near_end["action_is_pad"].sum()), "de 48 posiciones son relleno")

Un ejemplo de cada ocho tiene relleno: no es un detalle. Por eso existe la
máscara, y por eso un bloque más largo "desperdicia" más el final de cada
episodio.

## 3. La normalización

Antes de la red, el preprocesador lleva todo a una escala común (notebook L4):

- estado y acción: (x − media) / desvío, con las estadísticas del dataset;
- imagen: con la media y el desvío de ImageNet, porque así aprendió a ver la
  ResNet.

In [ ]:
batch_one = loaded.preprocessor({k: sample[k] for k in ["observation.state", CAMERA]})
print("estado crudo      :", np.round(sample["observation.state"][:4].numpy(), 3))
print("estado normalizado:", np.round(batch_one["observation.state"][0, :4].numpy(), 3))
print("imagen cruda: rango", float(sample[CAMERA].min()), "a", float(sample[CAMERA].max()),
      "→ normalizada: media", round(float(batch_one[CAMERA].mean()), 2), "desvío", round(float(batch_one[CAMERA].std()), 2))

## 4. El lote: lo que de verdad entra al `forward`

El entrenamiento no usa un ejemplo: usa un **lote** de ejemplos tomados al
azar de cualquier episodio. Un `DataLoader` de PyTorch los apila.

In [ ]:
loader = torch.utils.data.DataLoader(dataset, batch_size=8, shuffle=True, generator=torch.Generator().manual_seed(0))
batch = loaded.preprocessor(next(iter(loader)))
pd.DataFrame({k: {"forma": tuple(v.shape), "tipo": str(v.dtype)} for k, v in batch.items() if hasattr(v, "shape")}).T

Esto es exactamente lo que recibe `policy.forward(batch)`:

- `observation.images.cam_right_high`: (8, 3, 360, 640)
- `observation.state`: (8, 26)
- `action`: (8, 48, 26), la respuesta correcta (y, al entrenar, también la
  entrada del encoder del VAE)
- `action_is_pad`: (8, 48)

### Cuidado al separar entrenamiento y evaluación

En LeRobot 0.4.1, `LeRobotDataset(..., episodes=[...])` **no filtra** un
dataset local (L2). Para entrenar sólo con ciertos episodios hay que hacerlo a
mano: elegir los índices de sus cuadros y usar un `SubsetRandomSampler`. Así
se hace en L3.4.

## 5. El desequilibrio de la entrada

¿Cuántos números entran por ejemplo, y de dónde?

In [ ]:
counts = {"imagen (3×360×640)": 3 * 360 * 640, "estado (26)": 26}
target = config.chunk_size * 26
for name, n in counts.items():
    print(f"{name:20s} {n:>8,d} números")
print(f"{'respuesta (48×26)':20s} {target:>8,d} números")

La imagen es 26.000 veces más grande que el estado. Por eso la ResNet la
comprime primero a 240 tokens, y por eso el estado entra como un token propio:
si no, se perdería entre los píxeles (L3.3).

---

## Para profundizar: ejemplos vecinos casi iguales

22.251 ejemplos parecen muchos, pero dos ejemplos seguidos son casi el mismo:
la imagen cambia poco y el bloque está corrido un paso.

In [ ]:
chunks = np.stack([episode_actions[i:i + 48] for i in range(0, len(episode_actions) - 48)])
neighbour = np.abs(chunks[1:] - chunks[:-1]).mean()
random_pair = np.abs(chunks[np.random.permutation(len(chunks))] - chunks).mean()
print(f"diferencia media entre bloques vecinos: {neighbour:.4f} rad · entre bloques al azar del episodio: {random_pair:.4f} rad")

Los bloques vecinos son unas 80 veces más parecidos entre sí que dos al azar.
En la práctica el dataset tiene 61 demostraciones, no 22.000 ejemplos
independientes. Es una de las razones por las que la red puede memorizar.